# Module 03 — Offline Evaluations

Online evals (Module 2) grade live traffic. **Offline** evals grade the agent against a fixed **dataset**
so you can test a change *before* it ships. We'll evaluate the Northwind Mobile concierge at three
granularities — **final response (e2e)**, **single-step**, and **trajectory** — then A/B a few prompt
versions pulled from Context Hub.

> 📚 Docs: <https://docs.langchain.com/langsmith/evaluate-llm-application#run-the-evaluation>


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os, logging
logging.getLogger('asyncio').setLevel(logging.CRITICAL)
from collections import Counter
from typing import Any
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

from utils.config import project_for
from utils.models import model
from agents.telecom_concierge import build_telecom_concierge

username = input('Your name: ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')

client = Client()
# One agent, no checkpointer -> each dataset example is evaluated independently (no state bleed).
agent = build_telecom_concierge(with_checkpointer=False)
print('Project:', PROJECT_NAME)


## Part 1 — Dataset

A dataset is a versioned set of **examples**, each with `inputs` and reference `outputs`. We give every
example three reference fields so the *same* dataset drives all three eval types:

- `reference_answer` — a success **rubric** (not exact text) for the LLM judge
- `expected_first_tool` — the first tool we expect (single-step)
- `trajectory` — the full expected tool sequence (trajectory)

<img src="../images/dataset.png" alt="Dataset of examples" style="width: auto; max-height: 360px; border-radius: 8px;">

> **Two ways to build a dataset:** create it programmatically (below), **or** curate real traces into it
> via an **annotation queue** — route interesting/failing runs to reviewers who add them (and corrected
> outputs) to the dataset. *(Annotation-queue curation is demoed live.)*


In [ ]:
examples = [
    {'inputs': {'query': 'What plan and device does ACCT-1001 have?'},
     'outputs': {'reference_answer': 'States ACCT-1001 (Alex Rivera) is on Unlimited Premium with a Pixel 9 Pro.',
                 'expected_first_tool': 'get_customer_account', 'trajectory': ['get_customer_account']}},
    {'inputs': {'query': 'What does our late fee policy say?'},
     'outputs': {'reference_answer': 'Summarizes the late-fee policy (fee if unpaid past the due date; autopay avoids it).',
                 'expected_first_tool': 'search_support_docs', 'trajectory': ['search_support_docs']}},
    {'inputs': {'query': 'How much data has ACCT-1004 used this cycle, and are they over cap?'},
     'outputs': {'reference_answer': "Reports ACCT-1004's usage (~77 GB) and that Unlimited Premium has no cap.",
                 'expected_first_tool': 'get_data_usage', 'trajectory': ['get_data_usage']}},
    {'inputs': {'query': 'Pull up ACCT-1002 and tell me if they are close to their data cap.'},
     'outputs': {'reference_answer': 'Looks up ACCT-1002 (Jordan Lee, Unlimited Basic, 50 GB cap) and reports ~18.9 GB used — not near the cap.',
                 'expected_first_tool': 'get_customer_account', 'trajectory': ['get_customer_account', 'get_data_usage']}},
    {'inputs': {'query': 'ACCT-1005 says data is crawling in ZIP 94010 — triage it.'},
     'outputs': {'reference_answer': 'Checks network status for 94010 (partial outage) and delegates device troubleshooting to diagnostics; gives concrete next steps.',
                 'expected_first_tool': 'check_network_status', 'trajectory': ['check_network_status', 'task']}},
    {'inputs': {'query': 'What does account ACCT-9999 have?'},
     'outputs': {'reference_answer': 'Reports that ACCT-9999 is not found; does not fabricate any plan/device details.',
                 'expected_first_tool': 'get_customer_account', 'trajectory': ['get_customer_account']}},
]

dataset_name = f'{PROJECT_NAME}-evals'  # per-user: each attendee gets their own dataset
if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_id=client.read_dataset(dataset_name=dataset_name).id)
    print('Deleted existing dataset')

dataset = client.create_dataset(dataset_name)
client.create_examples(inputs=[e['inputs'] for e in examples],
                       outputs=[e['outputs'] for e in examples], dataset_id=dataset.id)
print(f'Created {dataset_name} with {len(examples)} examples')
print('View:', dataset.url)


## Part 2 — Final-response evaluation (end-to-end)

The broadest check: run the agent and judge the **final answer** against the rubric with an LLM-as-judge.

<img src="../images/final-response.png" alt="Final-response evaluation" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target: run the agent, return its final response."""
    config = {'configurable': {'thread_id': str(uuid7())}}
    result = agent.invoke({'messages': [{'role': 'user', 'content': inputs['query']}]}, config=config)
    return {'response': result['messages'][-1].content}


In [ ]:
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the rubric."""
    score: Annotated[bool, ..., 'True if the response meets the success rubric']
    reasoning: Annotated[str, ..., 'One short sentence of reasoning']

correctness_judge_prompt = """You are grading a Northwind Mobile telecom support assistant.

You will see the user's request, the assistant's final response, and a RUBRIC of success criteria.

Mark score=True if the response meets the criteria — even if worded differently. Mark score=False only if
it clearly missed the task, hallucinated facts (e.g. made-up plan prices, data caps, or roaming rates),
invented or leaked customer identity details, or refused without good reason.

Give one short sentence of reasoning either way.
"""

judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {'key': 'correctness', 'score': int(grade['score']), 'comment': grade['reasoning']}


In [ ]:
results = client.evaluate(
    run_agent_final, data=dataset_name, evaluators=[correctness_evaluator],
    experiment_prefix='final-response', max_concurrency=2,
)
print('Experiment:', results.experiment_name)


### Review in LangSmith

Open the **Datasets & Experiments** tab for this dataset to see the per-example scores; open an experiment
to compare inputs, outputs, and judge reasoning side by side.

<img src="../images/experiments_page.png" alt="Experiments page" style="width: auto; max-height: 360px; border-radius: 8px;">


## Part 3 — Single-step evaluation

A focused diagnostic: did the agent reach for the **right first tool**? Cheap, deterministic, and great for
catching routing regressions.

<img src="../images/single-step.png" alt="Single-step evaluation" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Tool-call names, in order."""
    names = []
    for msg in messages:
        if getattr(msg, 'tool_calls', None):
            names.extend(tc['name'] for tc in msg.tool_calls)
    return names

def run_agent_first_tool(inputs: dict) -> dict:
    config = {'configurable': {'thread_id': str(uuid7())}}
    result = agent.invoke({'messages': [{'role': 'user', 'content': inputs['query']}]}, config=config)
    calls = extract_tool_calls(result['messages'])
    return {'first_tool': calls[0] if calls else None}

def first_tool_match(outputs, reference_outputs):
    return {'key': 'first_tool_match',
            'score': int(outputs['first_tool'] == reference_outputs['expected_first_tool'])}


In [ ]:
results = client.evaluate(
    run_agent_first_tool, data=dataset_name, evaluators=[first_tool_match],
    experiment_prefix='single-step', max_concurrency=2,
)
print('Experiment:', results.experiment_name)


## Part 4 — Trajectory evaluation

The full picture: did the agent take the **expected sequence of steps**? We score exact match plus extra /
missing steps.

<img src="../images/trajectory.png" alt="Trajectory evaluation" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
def run_agent_trajectory(inputs: dict) -> dict:
    config = {'configurable': {'thread_id': str(uuid7())}}
    result = agent.invoke({'messages': [{'role': 'user', 'content': inputs['query']}]}, config=config)
    return {'trajectory': extract_tool_calls(result['messages'])}

def trajectory_match(outputs, reference_outputs):
    return {'key': 'exact_match', 'score': int(outputs['trajectory'] == reference_outputs['trajectory'])}

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs['trajectory']) - Counter(reference_outputs['trajectory'])
    return {'key': 'extra_steps', 'score': sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs['trajectory']) - Counter(outputs['trajectory'])
    return {'key': 'missing_steps', 'score': sum(missing.values())}


In [ ]:
results = client.evaluate(
    run_agent_trajectory, data=dataset_name, evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix='trajectory', max_concurrency=2,
)
print('Experiment:', results.experiment_name)


## Part 5 — A/B testing prompt versions (from Context Hub)

Offline evals shine for comparing changes. Here we keep a few **system-prompt versions in Context Hub**,
run the same e2e evaluation against each, and tag every run with **experiment metadata** so they're easy to
compare. (You could also use tagged commits of one agent repo; here each variant is its own repo.)

- `baseline` — the hallucination-prone prompt (guesses figures from memory)
- `grounded` — only states figures from tools/docs; never reads back full PII
- `concise` — grounded **and** terse

<img src="../images/experiment_comparison.png" alt="Compare experiments" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
from utils.context_hub import push_agent_variant, pull_agent_variant

PROMPT_VARIANTS = {
    'baseline': ('You are the Northwind Mobile concierge for a call-center rep. Answer quickly and '
                 'confidently. If the docs lack an exact figure (price, speed, promo), give your best '
                 'number from memory rather than hedging.'),
    'grounded': ('You are the Northwind Mobile concierge for a call-center rep. Only state figures '
                 '(prices, data caps, speeds) that come from a tool or the support docs. If you do not '
                 'have it, say so and offer to look it up. Never read back full SSNs or card numbers.'),
    'concise': ('You are the Northwind Mobile concierge for a call-center rep. Only state figures that '
                'come from tools/docs; never invent prices or read back full SSNs/cards. Keep answers to '
                '1-2 sentences.'),
}

# Store each variant in Context Hub.
for _name, _content in PROMPT_VARIANTS.items():
    print(_name, '->', push_agent_variant(_name, _content))


In [ ]:
ab_experiments = {}
for name in PROMPT_VARIANTS:
    prompt = pull_agent_variant(name, fallback=PROMPT_VARIANTS[name])  # pull the version from Context Hub
    ab_agent = build_telecom_concierge(system_prompt=prompt, with_checkpointer=False)

    def run_variant(inputs, _agent=ab_agent):
        config = {'configurable': {'thread_id': str(uuid7())}}
        r = _agent.invoke({'messages': [{'role': 'user', 'content': inputs['query']}]}, config=config)
        return {'response': r['messages'][-1].content}

    results = client.evaluate(
        run_variant, data=dataset_name, evaluators=[correctness_evaluator],
        experiment_prefix=f'e2e-{name}',
        metadata={'prompt_version': name, 'source': 'context-hub'},  # experiment metadata
        max_concurrency=2,
    )
    ab_experiments[name] = results.experiment_name
    print(f'{name:9s} -> {results.experiment_name}')


### Compare and set a baseline

In **Datasets & Experiments**, select the three `e2e-*` experiments to compare correctness side by side —
filter/group by the `prompt_version` metadata. Set the winner as the **baseline** so future runs diff
against it.

<img src="../images/set_baseline.png" alt="Set an experiment as baseline" style="width: auto; max-height: 360px; border-radius: 8px;">


## Recap

| Eval type | Target returns | Evaluator(s) | Reference field |
|---|---|---|---|
| Final response (e2e) | final answer | `correctness` (LLM judge) | `reference_answer` |
| Single-step | first tool called | `first_tool_match` | `expected_first_tool` |
| Trajectory | tool sequence | `exact_match`, `extra_steps`, `missing_steps` | `trajectory` |
| A/B (e2e) | final answer per prompt version | `correctness` + experiment `metadata` | `reference_answer` |

Datasets can be built programmatically **or** curated from traces via an annotation queue. Prompt versions
live in Context Hub and are compared with experiment metadata.
